# ID3 — learn from school activity data

Supervised classification: known yes/no answers teach the tree which questions to ask.

All data and code are prepared. **Run ▶ the code cells in order**. Change one number and rerun to investigate. JupyterLite uses Python in your browser; no account is required.


## 1. See the training records


In [ ]:
from math import log2
from collections import Counter
columns=['sky','wind','temperature','go']
data=[
 ('sunny','calm','warm','yes'), ('sunny','calm','cold','yes'),
 ('sunny','windy','warm','yes'), ('sunny','windy','cold','yes'),
 ('cloudy','calm','warm','yes'), ('cloudy','calm','cold','yes'),
 ('cloudy','windy','warm','no'), ('cloudy','windy','cold','no'),
 ('rainy','calm','warm','no'), ('rainy','calm','cold','no'),
 ('rainy','windy','warm','no'), ('rainy','windy','cold','no')]
rows=[dict(zip(columns,row)) for row in data]
def entropy(values):
    counts=Counter(values); n=len(values)
    return -sum((v/n)*log2(v/n) for v in counts.values())
print('Records:',len(rows),'Known labels:',dict(Counter(row['go'] for row in rows)))


## 2. Compare information gain


In [ ]:
def gain(dataset,feature):
    before=entropy([row['go'] for row in dataset])
    remainder=0
    for value in sorted({row[feature] for row in dataset}):
        group=[row for row in dataset if row[feature]==value]
        yes=sum(row['go']=='yes' for row in group)
        h=entropy([row['go'] for row in group])
        print(feature,'=',value,'→ [',yes,'+,',len(group)-yes,'−], H =',round(h,3))
        remainder+=len(group)/len(dataset)*h
    return before-remainder
for f in columns[:-1]:
    print('Information gain for',f,'=',round(gain(rows,f),3),'bits')


## 3. Train and test an ID3 tree


In [ ]:
from collections import Counter
def train(examples,features):
    labels=[row['go'] for row in examples]
    majority=sorted(Counter(labels).items(), key=lambda t:(-t[1],t[0]))[0][0]
    if len(set(labels))==1:return labels[0]
    if not features:return majority
    best=max(features,key=lambda f: gain_silent(examples,f))
    branches={}
    for value in sorted({row[best] for row in examples}):
        subset=[row for row in examples if row[best]==value]
        branches[value]=train(subset,[f for f in features if f!=best])
    return {'feature':best,'branches':branches,'fallback':majority}
def gain_silent(examples,feature):
    groups=[[r for r in examples if r[feature]==v]
            for v in sorted({r[feature] for r in examples})]
    return entropy([r['go'] for r in examples])-sum(
        len(group)/len(examples)*entropy([r['go'] for r in group]) for group in groups)
def predict(tree,example):
    path=[]
    while isinstance(tree,dict):
        feat=tree['feature']; value=example.get(feat)
        path.append(f'{feat} = {value}')
        if value not in tree['branches']:
            path.append('WARNING: unseen value, using majority fallback')
            return tree['fallback'],path
        tree=tree['branches'][value]
    path.append('predict '+tree)
    return tree,path
tree=train(rows,columns[:-1])
print('Learned tree:',tree)
for example in [
 {'sky':'cloudy','wind':'calm','temperature':'cold'},
 {'sky':'rainy','wind':'windy','temperature':'warm'},
 {'sky':'foggy','wind':'calm','temperature':'warm'}]:
 result,path=predict(tree,example)
 print(' → '.join(path),'→',result)


## Try it yourself

Change one label in the data. Predict whether the information gains and the tree will change. Run the cells again.
